# 中证800 V66：滚动训练生产规则综合验证（Ensemble / Agreement / 防守机制）

V65 回答了“滚动训练方法论是否有 OOS 超额”：结论是有，但训练长度和 regime/cutoff 都很关键，短窗口不稳，2023/2025-03/2025-05/2026-03 是典型失效期。

V66 不再训练新模型，而是在 V65 的滚动训练结果上做**生产规则层面的综合验证**：

- 单模型基线：expanding / rolling60 / rolling72
- 模型融合：expanding + rolling60，或所有可用方法 blend
- 一致性规则：交集优先、分歧时降持仓数量、低一致性空仓/半仓
- 失效诊断：分歧度、score 相关性、target overlap 是否能解释坏月
- JQ-like 日频账户路径：如果聚宽 `get_price` 可用，自动跑真实账户近似

核心问题：

> 如果真实上线，我们应该只用 rolling60，还是 expanding + rolling60 ensemble？分歧时是否应该降仓？

本 notebook 默认复用 V65 输出，尤其是 `v65_score_panel.csv`。如果该文件不存在，请先完整运行 V65。


In [ ]:
import os
import ast
import warnings
from pathlib import Path
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 240)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def progress_iter(iterable, total=None, desc="progress", leave=True):
    if tqdm is not None:
        return tqdm(iterable, total=total, desc=desc, leave=leave)
    def _gen():
        every = max(1, int((total or 100) / 20))
        for i, item in enumerate(iterable, 1):
            if i == 1 or i % every == 0 or (total is not None and i == total):
                print("%s %s%s" % (desc, i, "/%s" % total if total else ""))
            yield item
    return _gen()

# =========================
# Config
# =========================
V65_REQUIRED_FILES = ["v65_score_panel.csv", "v65_rolling_oos_monthly.csv", "v65_fold_plan.csv"]
V65_OUT_DIR_CANDIDATES = [
        Path("csi800_ml_v65_rolling_retrain_methodology_outputs"),
        Path.home() / "Downloads" / "csi800_ml_v65_rolling_retrain_methodology_outputs",
        Path.home() / "Downloads",
    ]

def resolve_v65_out_dir() -> Path:
    best_dir = V65_OUT_DIR_CANDIDATES[0]
    best_count = -1
    for d in V65_OUT_DIR_CANDIDATES:
        hit_count = sum((d / f).exists() for f in V65_REQUIRED_FILES)
        if hit_count == len(V65_REQUIRED_FILES):
            return d
        if hit_count > best_count:
            best_dir = d
            best_count = hit_count
    return best_dir

V65_OUT_DIR = resolve_v65_out_dir()
OUT_DIR = Path("csi800_ml_v66_production_rule_ensemble_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "alpha_1m"
STOCK_COL = "stock"
DATE_COL = "rebalance_date"
SCORE_COL = "score"
BENCHMARK = "000906.XSHG"

PORTFOLIO_RULE = "top8_board_cap"
STOCK_NUM = 8
BOARD_CAPS = {"chinext": 3, "star": 2}
BOARD_CAPS_TEXT = ";".join(["%s:%s" % (k, BOARD_CAPS[k]) for k in sorted(BOARD_CAPS)])

BASE_POLICIES = ["expanding_min36", "rolling60m"]
ALL_POLICY_CANDIDATES = ["expanding_min36", "rolling60m", "rolling72m"]
WEAK_POLICY_CANDIDATES = ["rolling36m", "rolling48m"]
INCLUDE_WEAK_BASELINES = True

RANDOM_SIM_N = 500
RANDOM_SEED = 42

SLIPPAGE_RATE = 0.00246
OPEN_COMMISSION = 0.0003
CLOSE_COMMISSION = 0.0003
CLOSE_TAX = 0.001
BUY_COST_RATE = SLIPPAGE_RATE + OPEN_COMMISSION
SELL_COST_RATE = SLIPPAGE_RATE + CLOSE_COMMISSION + CLOSE_TAX

RUN_JQ_LIKE_SIMULATOR = "AUTO"
JQ_SIM_INITIAL_CASH = 300000.0
JQ_SIM_PRICE_FQ = None
JQ_SIM_BUY_TIME_FIELD = "open"
JQ_SIM_MARK_FIELD = "close"
JQ_SIM_CHUNK_SIZE = 120
MIN_COMMISSION = 5.0
NORMAL_MIN_LOT = 100
KCB_MIN_LOT = 200

print("V65_OUT_DIR:", V65_OUT_DIR.resolve())
print("OUT_DIR:", OUT_DIR.resolve())
print("base policies:", BASE_POLICIES)
print("portfolio:", PORTFOLIO_RULE, STOCK_NUM, BOARD_CAPS_TEXT)


In [ ]:
def read_csv_required(path: Path) -> pd.DataFrame:
    if not path.exists():
        missing = [f for f in V65_REQUIRED_FILES if not (V65_OUT_DIR / f).exists()]
        searched = [str(d.resolve()) for d in V65_OUT_DIR_CANDIDATES]
        raise IOError(
            "missing required file: %s\n"
            "V65_OUT_DIR resolved to: %s\n"
            "missing in that directory: %s\n"
            "searched directories: %s\n"
            "请先完整运行最新版 V65；V66 的融合/交集/防守规则需要逐股票逐月 score panel。"
            % (path, V65_OUT_DIR.resolve(), missing, searched)
        )
    df = pd.read_csv(path)
    print("loaded", path.name, df.shape)
    return df


score_panel_df = read_csv_required(V65_OUT_DIR / "v65_score_panel.csv")
v65_monthly_df = read_csv_required(V65_OUT_DIR / "v65_rolling_oos_monthly.csv")
fold_plan_df = read_csv_required(V65_OUT_DIR / "v65_fold_plan.csv")

for df in [score_panel_df, v65_monthly_df, fold_plan_df]:
    for c in [DATE_COL, "next_date", "train_start", "train_end", "test_start", "test_end"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce").dt.normalize()

if SCORE_COL not in score_panel_df.columns:
    raise ValueError("v65_score_panel.csv missing score column")
if TARGET_COL not in score_panel_df.columns:
    raise ValueError("v65_score_panel.csv missing target column: " + TARGET_COL)

RAW_RET_COL = None
for c in ["raw_return_1m", "stock_return_1m", "return_1m", "next_return_1m", "gross_raw_ret"]:
    if c in score_panel_df.columns:
        RAW_RET_COL = c
        break
BENCH_RET_COL = None
for c in ["benchmark_csi800_1m", "benchmark_000906_1m", "benchmark_alla_1m", "cum_csi800_1m", "benchmark_ret"]:
    if c in score_panel_df.columns:
        BENCH_RET_COL = c
        break

available_policies = sorted(score_panel_df["train_policy"].astype(str).unique())
print("available policies:", available_policies)
print("raw:", RAW_RET_COL, "bench:", BENCH_RET_COL, "target:", TARGET_COL)
print("date range:", score_panel_df[DATE_COL].min(), score_panel_df[DATE_COL].max())


In [ ]:
def board_type(stock):
    s = str(stock)
    if s.startswith("30"):
        return "chinext"
    if s.startswith(("688", "689")):
        return "star"
    return "main"


def board_cap_allows(selected, stock, board_caps):
    if not board_caps:
        return True
    b = board_type(stock)
    if b not in board_caps:
        return True
    return sum(1 for x in selected if board_type(x) == b) < int(board_caps[b])


def build_board_capped_targets(sorted_stocks, target_num=STOCK_NUM, board_caps=BOARD_CAPS):
    selected = []
    for stock in sorted_stocks:
        s = str(stock)
        if s in selected:
            continue
        if board_cap_allows(selected, s, board_caps):
            selected.append(s)
        if len(selected) >= target_num:
            return selected[:target_num]
    for stock in sorted_stocks:
        s = str(stock)
        if s not in selected:
            selected.append(s)
        if len(selected) >= target_num:
            break
    return selected[:target_num]


def summarize_target_board(targets):
    out = {"board_main": 0, "board_chinext": 0, "board_star": 0}
    for stock in targets:
        b = board_type(stock)
        out["board_" + b] = out.get("board_" + b, 0) + 1
    n = max(1, len(targets))
    out["board_main_ratio"] = out.get("board_main", 0) / float(n)
    out["board_chinext_ratio"] = out.get("board_chinext", 0) / float(n)
    out["board_star_ratio"] = out.get("board_star", 0) / float(n)
    out["board_hhi"] = sum((out[k] / float(n)) ** 2 for k in ["board_main", "board_chinext", "board_star"])
    return out


def calc_nav(ret_series):
    s = pd.Series(ret_series).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return (1.0 + s).cumprod() if len(s) else pd.Series(dtype=float)


def calc_drawdown_from_returns(ret_series):
    nav = calc_nav(ret_series)
    if len(nav) == 0:
        return np.nan
    return float((nav / nav.cummax() - 1.0).min())


def summarize_return_series(ret_series):
    s = pd.Series(ret_series).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return {"months": 0, "cum_ret": np.nan, "ann_ret": np.nan, "mean_ret": np.nan, "win_rate": np.nan, "max_drawdown": np.nan, "sharpe12": np.nan}
    std = s.std(ddof=1)
    return {
        "months": int(len(s)),
        "cum_ret": float((1.0 + s).prod() - 1.0),
        "ann_ret": float((1.0 + s).prod() ** (12.0 / len(s)) - 1.0) if len(s) else np.nan,
        "mean_ret": float(s.mean()),
        "win_rate": float((s > 0).mean()),
        "max_drawdown": calc_drawdown_from_returns(s),
        "sharpe12": float(s.mean() / std * np.sqrt(12)) if len(s) > 1 and std > 0 else np.nan,
    }


def safe_rank_ic(a, b):
    s = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3 or s["a"].nunique() < 2 or s["b"].nunique() < 2:
        return np.nan
    return s["a"].rank(pct=True).corr(s["b"].rank(pct=True))


def stable_text_seed(text):
    total = 0
    for i, ch in enumerate(str(text)):
        total += (i + 1) * ord(ch)
    return int(total % 100000)


def get_return_map(month_df, col):
    if col is None or col not in month_df.columns:
        return {}
    return dict(zip(month_df[STOCK_COL].astype(str), pd.to_numeric(month_df[col], errors="coerce")))


def calc_portfolio_return_from_map(ret_map, stocks):
    vals = []
    for stock in stocks:
        v = ret_map.get(str(stock), np.nan)
        if not pd.isnull(v):
            vals.append(float(v))
    return float(np.mean(vals)) if vals else np.nan


def calc_equal_weight_turnover(prev_targets, targets):
    targets = list(targets)
    if len(targets) == 0:
        return 0.0, 0.0, 0
    if prev_targets is None:
        return 1.0, 0.0, 0
    overlap = len(set(prev_targets).intersection(set(targets)))
    denom = float(max(1, max(len(prev_targets), len(targets))))
    one_way = 1.0 - overlap / denom
    return float(one_way), float(one_way), int(overlap)


def calc_trade_cost(prev_targets, targets):
    buy_turnover, sell_turnover, overlap = calc_equal_weight_turnover(prev_targets, targets)
    cost = buy_turnover * BUY_COST_RATE + sell_turnover * SELL_COST_RATE
    if prev_targets is None:
        cost = buy_turnover * BUY_COST_RATE
    return float(cost), float(buy_turnover), float(sell_turnover), int(overlap)


In [ ]:
def month_policy_panel(dt, policy):
    d = score_panel_df[(score_panel_df[DATE_COL] == pd.Timestamp(dt)) & (score_panel_df["train_policy"].astype(str) == str(policy))].copy()
    d = d.dropna(subset=[SCORE_COL, TARGET_COL])
    d[STOCK_COL] = d[STOCK_COL].astype(str)
    return d


def normalize_scores_by_policy(policy_frames):
    merged = None
    for policy, df in policy_frames.items():
        x = df[[STOCK_COL, SCORE_COL]].copy()
        x = x.rename(columns={SCORE_COL: "score__" + policy})
        if merged is None:
            merged = x
        else:
            merged = merged.merge(x, on=STOCK_COL, how="outer")
    if merged is None:
        return pd.DataFrame()
    for policy in policy_frames:
        col = "score__" + policy
        rcol = "rank__" + policy
        zcol = "z__" + policy
        merged[rcol] = merged[col].rank(pct=True)
        s = pd.to_numeric(merged[col], errors="coerce")
        std = s.std(ddof=0)
        merged[zcol] = (s - s.mean()) / std if std and std > 0 else np.nan
    return merged


def build_blend_scores(dt, policies):
    frames = {p: month_policy_panel(dt, p) for p in policies if p in available_policies}
    frames = {p: df for p, df in frames.items() if not df.empty}
    if not frames:
        return pd.DataFrame(), {}
    merged = normalize_scores_by_policy(frames)
    rank_cols = ["rank__" + p for p in frames]
    z_cols = ["z__" + p for p in frames]
    merged["blend_rank_score"] = merged[rank_cols].mean(axis=1, skipna=True)
    merged["blend_z_score"] = merged[z_cols].mean(axis=1, skipna=True)
    merged["available_policy_count"] = merged[rank_cols].notnull().sum(axis=1)
    return merged, frames


def base_targets_for_policy(dt, policy, target_num=STOCK_NUM):
    d = month_policy_panel(dt, policy)
    if d.empty:
        return []
    sorted_stocks = list(d.sort_values(SCORE_COL, ascending=False)[STOCK_COL].astype(str))
    return build_board_capped_targets(sorted_stocks, target_num, BOARD_CAPS)


def blend_targets(dt, policies, target_num=STOCK_NUM, min_policy_count=1):
    merged, frames = build_blend_scores(dt, policies)
    if merged.empty:
        return [], merged, frames
    x = merged[merged["available_policy_count"] >= int(min_policy_count)].copy()
    if x.empty:
        x = merged.copy()
    sorted_stocks = list(x.sort_values(["blend_rank_score", "blend_z_score"], ascending=False)[STOCK_COL].astype(str))
    return build_board_capped_targets(sorted_stocks, target_num, BOARD_CAPS), merged, frames


def intersection_fill_targets(dt, policies, target_num=STOCK_NUM, min_intersection=1):
    base = {p: base_targets_for_policy(dt, p, STOCK_NUM) for p in policies if p in available_policies}
    base = {p: t for p, t in base.items() if len(t)}
    if not base:
        return [], {}, pd.DataFrame(), {}
    sets = [set(t) for t in base.values()]
    inter = set.intersection(*sets) if sets else set()
    merged, frames = build_blend_scores(dt, list(base.keys()))
    if merged.empty:
        return [], base, merged, frames
    score_order = list(merged.sort_values(["blend_rank_score", "blend_z_score"], ascending=False)[STOCK_COL].astype(str))
    targets = []
    if len(inter) >= min_intersection:
        inter_order = [s for s in score_order if s in inter]
        targets.extend(build_board_capped_targets(inter_order, min(len(inter_order), target_num), BOARD_CAPS))
    for s in score_order:
        if len(targets) >= target_num:
            break
        if s in targets:
            continue
        if board_cap_allows(targets, s, BOARD_CAPS):
            targets.append(s)
    return targets[:target_num], base, merged, frames


def policy_agreement_metrics(dt, p1="expanding_min36", p2="rolling60m"):
    t1 = base_targets_for_policy(dt, p1, STOCK_NUM)
    t2 = base_targets_for_policy(dt, p2, STOCK_NUM)
    s1, s2 = set(t1), set(t2)
    overlap = len(s1 & s2) if t1 and t2 else np.nan
    union = len(s1 | s2) if t1 and t2 else np.nan
    jaccard = overlap / float(union) if union and union > 0 else np.nan
    d1 = month_policy_panel(dt, p1)
    d2 = month_policy_panel(dt, p2)
    corr = np.nan
    if not d1.empty and not d2.empty:
        x = d1[[STOCK_COL, SCORE_COL]].rename(columns={SCORE_COL:"s1"}).merge(d2[[STOCK_COL, SCORE_COL]].rename(columns={SCORE_COL:"s2"}), on=STOCK_COL, how="inner")
        if len(x) >= 10:
            corr = safe_rank_ic(x["s1"], x["s2"])
    return {"expanding_target_count": len(t1), "rolling60_target_count": len(t2), "agreement_overlap": overlap, "agreement_jaccard": jaccard, "score_rank_corr_exp_roll60": corr}


## 规则清单

这里的规则分三类：

1. 单方法基线：用于确认 V65 结论是否在同一套评估函数里复现。
2. 融合规则：只用 ex-ante score / target 一致性，不看未来收益。
3. 防守规则：当 expanding 与 rolling60 分歧大时，主动降低持仓数量或空仓。

规则不使用未来收益，也不使用当月真实 RankIC/random percentile 做决策。


In [ ]:
def build_rule_targets(dt, rule_name):
    # Single-policy baselines.
    if rule_name.startswith("single__"):
        policy = rule_name.split("__", 1)[1]
        return base_targets_for_policy(dt, policy, STOCK_NUM), {"rule_family": "single", "source_policies": policy}

    if rule_name == "blend_exp_roll60_top8":
        t, merged, frames = blend_targets(dt, ["expanding_min36", "rolling60m"], STOCK_NUM, min_policy_count=1)
        return t, {"rule_family": "blend", "source_policies": "expanding_min36,rolling60m"}

    if rule_name == "intersect_fill_exp_roll60_top8":
        t, base, merged, frames = intersection_fill_targets(dt, ["expanding_min36", "rolling60m"], STOCK_NUM, min_intersection=1)
        return t, {"rule_family": "intersection_fill", "source_policies": "expanding_min36,rolling60m"}

    if rule_name == "agreement_adaptive_exp_roll60":
        agree = policy_agreement_metrics(dt)
        overlap = agree.get("agreement_overlap", np.nan)
        if pd.isnull(overlap):
            t, _, _, _ = intersection_fill_targets(dt, ["expanding_min36", "rolling60m"], STOCK_NUM, min_intersection=0)
            target_num = STOCK_NUM
            mode = "fallback_top8"
        elif overlap >= 5:
            target_num = 8
            mode = "high_agreement_top8"
            t, _, _, _ = intersection_fill_targets(dt, ["expanding_min36", "rolling60m"], target_num, min_intersection=1)
        elif overlap >= 3:
            target_num = 6
            mode = "mid_agreement_top6"
            t, _, _, _ = intersection_fill_targets(dt, ["expanding_min36", "rolling60m"], target_num, min_intersection=1)
        else:
            target_num = 4
            mode = "low_agreement_top4"
            t, _, _, _ = intersection_fill_targets(dt, ["expanding_min36", "rolling60m"], target_num, min_intersection=0)
        meta = {"rule_family": "agreement_adaptive", "source_policies": "expanding_min36,rolling60m", "agreement_mode": mode, "adaptive_target_num": target_num}
        meta.update(agree)
        return t, meta

    if rule_name == "low_agreement_cash_or_top4":
        agree = policy_agreement_metrics(dt)
        overlap = agree.get("agreement_overlap", np.nan)
        if not pd.isnull(overlap) and overlap < 3:
            t = []
            mode = "cash"
        else:
            t, _, _, _ = intersection_fill_targets(dt, ["expanding_min36", "rolling60m"], 8, min_intersection=1)
            mode = "normal_top8"
        meta = {"rule_family": "defensive_cash", "source_policies": "expanding_min36,rolling60m", "agreement_mode": mode, "adaptive_target_num": len(t)}
        meta.update(agree)
        return t, meta

    if rule_name == "blend_all_available_top8":
        policies = [p for p in ALL_POLICY_CANDIDATES if p in available_policies]
        t, merged, frames = blend_targets(dt, policies, STOCK_NUM, min_policy_count=1)
        return t, {"rule_family": "blend_all", "source_policies": ",".join(policies)}

    if rule_name == "intersect_fill_all_available_top8":
        policies = [p for p in ALL_POLICY_CANDIDATES if p in available_policies]
        t, base, merged, frames = intersection_fill_targets(dt, policies, STOCK_NUM, min_intersection=1)
        return t, {"rule_family": "intersection_fill_all", "source_policies": ",".join(policies)}

    raise ValueError("unknown rule: " + str(rule_name))


RULE_SPECS = [
    "single__expanding_min36",
    "single__rolling60m",
    "single__rolling72m",
    "blend_exp_roll60_top8",
    "intersect_fill_exp_roll60_top8",
    "agreement_adaptive_exp_roll60",
    "low_agreement_cash_or_top4",
    "blend_all_available_top8",
    "intersect_fill_all_available_top8",
]
if INCLUDE_WEAK_BASELINES:
    RULE_SPECS.extend(["single__rolling36m", "single__rolling48m"])

# Keep only rules whose required policies are possible; unavailable single rules are skipped dynamically.
print("rules:", RULE_SPECS)


In [ ]:
def representative_month_frame(dt, policies=None):
    policies = policies or ["expanding_min36", "rolling60m", "rolling72m", "rolling36m", "rolling48m"]
    for p in policies:
        d = month_policy_panel(dt, p)
        if not d.empty:
            return d.copy()
    return pd.DataFrame()


def random_percentile_for_targets(month_df, targets, ret_col, seed_key):
    if ret_col is None or ret_col not in month_df.columns or not targets:
        return np.nan
    actual_ret = calc_portfolio_return_from_map(get_return_map(month_df, ret_col), targets)
    if pd.isnull(actual_ret):
        return np.nan
    d = month_df.dropna(subset=[ret_col]).copy().reset_index(drop=True)
    stocks = d[STOCK_COL].astype(str).values
    rets = pd.to_numeric(d[ret_col], errors="coerce").replace([np.inf, -np.inf], np.nan).values.astype(float)
    valid = np.isfinite(rets)
    stocks = stocks[valid]
    rets = rets[valid]
    if len(rets) == 0:
        return np.nan
    stock_to_idx = {s: i for i, s in enumerate(stocks)}
    base_idx = np.arange(len(stocks), dtype=np.int32)
    rng = np.random.RandomState(seed_key)
    vals = np.empty(int(RANDOM_SIM_N), dtype=float)
    target_num = max(1, len(targets))
    for i in range(int(RANDOM_SIM_N)):
        perm = rng.permutation(base_idx)
        picked = build_board_capped_targets([stocks[j] for j in perm], target_num, BOARD_CAPS)
        picked_idx = [stock_to_idx[s] for s in picked if s in stock_to_idx]
        vals[i] = float(np.nanmean(rets[picked_idx])) if len(picked_idx) else np.nan
    vals = pd.Series(vals).replace([np.inf, -np.inf], np.nan).dropna()
    return float((vals <= actual_ret).mean()) if len(vals) else np.nan


def evaluate_rule_monthly(rule_name, months):
    rows = []
    prev_targets = None
    for dt in progress_iter(months, total=len(months), desc="months %s" % rule_name, leave=False):
        dt = pd.Timestamp(dt)
        targets, meta = build_rule_targets(dt, rule_name)
        month_df = representative_month_frame(dt, meta.get("source_policies", "").split(",") if meta.get("source_policies") else None)
        if month_df.empty:
            continue
        raw_ret_map = get_return_map(month_df, RAW_RET_COL)
        alpha_ret_map = get_return_map(month_df, TARGET_COL)
        gross_alpha_ret = calc_portfolio_return_from_map(alpha_ret_map, targets) if targets else 0.0
        gross_raw_ret = calc_portfolio_return_from_map(raw_ret_map, targets) if targets and RAW_RET_COL else np.nan
        cost, buy_turnover, sell_turnover, overlap_prev = calc_trade_cost(prev_targets, targets)
        proxy_net_alpha_ret = gross_alpha_ret - cost if len(targets) else 0.0
        if not pd.isnull(gross_raw_ret):
            proxy_net_raw_ret = (1.0 + gross_raw_ret) * (1.0 - cost) - 1.0
        else:
            proxy_net_raw_ret = np.nan
        proxy_net_excess_ret = proxy_net_alpha_ret
        real_sorted = month_df.sort_values(TARGET_COL, ascending=False)
        top10 = set(real_sorted.head(10)[STOCK_COL].astype(str))
        top20 = set(real_sorted.head(20)[STOCK_COL].astype(str))
        selected_rows = month_df.set_index(STOCK_COL).reindex(targets) if len(targets) else pd.DataFrame()
        board = summarize_target_board(targets) if len(targets) else {"board_main": 0, "board_chinext": 0, "board_star": 0, "board_main_ratio": 0.0, "board_chinext_ratio": 0.0, "board_star_ratio": 0.0, "board_hhi": 0.0}
        agree = policy_agreement_metrics(dt) if "expanding_min36" in available_policies and "rolling60m" in available_policies else {}
        seed_key = int(dt.strftime("%Y%m%d")) + stable_text_seed(rule_name)
        row = {
            "rule_name": rule_name,
            "rule_family": meta.get("rule_family", ""),
            "source_policies": meta.get("source_policies", ""),
            "agreement_mode": meta.get("agreement_mode", ""),
            "adaptive_target_num": meta.get("adaptive_target_num", len(targets)),
            DATE_COL: dt,
            "next_date": month_df["next_date"].iloc[0] if "next_date" in month_df.columns else pd.NaT,
            "target_count": len(targets),
            "targets": ",".join(targets),
            "gross_alpha_ret": gross_alpha_ret,
            "gross_raw_ret": gross_raw_ret,
            "trade_cost": cost,
            "buy_turnover": buy_turnover,
            "sell_turnover": sell_turnover,
            "target_overlap_prev": overlap_prev,
            "proxy_net_alpha_ret": proxy_net_alpha_ret,
            "proxy_net_raw_ret": proxy_net_raw_ret,
            "proxy_net_excess_ret": proxy_net_excess_ret,
            "rank_ic_blend_or_primary": safe_rank_ic(month_df[SCORE_COL], month_df[TARGET_COL]) if SCORE_COL in month_df.columns else np.nan,
            "selected_hit_real_top10": len(set(targets) & top10),
            "selected_hit_real_top20": len(set(targets) & top20),
            "selected_hit_rate_real_top20": len(set(targets) & top20) / float(max(1, len(targets))) if len(targets) else 0.0,
            "target_avg_realized_rank": float(selected_rows["realized_rank_pct"].mean()) if "realized_rank_pct" in selected_rows.columns and len(selected_rows) else np.nan,
            "random_alpha_percentile": random_percentile_for_targets(month_df, targets, TARGET_COL, seed_key),
        }
        row.update(board)
        row.update(agree)
        row.update({k: v for k, v in meta.items() if k not in row})
        rows.append(row)
        prev_targets = list(targets)
    return pd.DataFrame(rows)


all_months = sorted(pd.to_datetime(score_panel_df[DATE_COL].dropna().unique()))
print("all score months:", len(all_months), pd.Timestamp(all_months[0]).date(), pd.Timestamp(all_months[-1]).date())

parts = []
for rule in progress_iter(RULE_SPECS, total=len(RULE_SPECS), desc="rules"):
    if rule.startswith("single__"):
        p = rule.split("__", 1)[1]
        if p not in available_policies:
            print("skip unavailable", rule)
            continue
    df = evaluate_rule_monthly(rule, all_months)
    if not df.empty:
        parts.append(df)

rule_monthly_df = pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame()
if rule_monthly_df.empty:
    raise ValueError("empty rule_monthly_df")
rule_monthly_df = rule_monthly_df.sort_values(["rule_name", DATE_COL]).reset_index(drop=True)
rule_monthly_df["proxy_nav"] = rule_monthly_df.groupby("rule_name")["proxy_net_excess_ret"].transform(lambda s: calc_nav(s).values if len(s) else s)
rule_monthly_df["proxy_drawdown"] = rule_monthly_df.groupby("rule_name")["proxy_net_excess_ret"].transform(lambda s: (calc_nav(s) / calc_nav(s).cummax() - 1.0).values if len(s) else s)

display(rule_monthly_df.tail(30))


In [ ]:
def summarize_rule(g, ret_col="proxy_net_excess_ret"):
    g = g.sort_values(DATE_COL).copy()
    st = summarize_return_series(g[ret_col])
    return pd.Series({
        "rule_name": g["rule_name"].iloc[0],
        "rule_family": g["rule_family"].iloc[0],
        "source_policies": g["source_policies"].iloc[0],
        "months": st["months"],
        "start": g[DATE_COL].min(),
        "end": g[DATE_COL].max(),
        "cum_ret": st["cum_ret"],
        "ann_ret": st["ann_ret"],
        "mean_monthly_ret": st["mean_ret"],
        "win_rate": st["win_rate"],
        "max_drawdown": st["max_drawdown"],
        "sharpe12": st["sharpe12"],
        "worst_month": float(pd.to_numeric(g[ret_col], errors="coerce").min()),
        "avg_target_count": float(g["target_count"].mean()),
        "avg_random_alpha_percentile": float(g["random_alpha_percentile"].mean()),
        "p25_random_alpha_percentile": float(g["random_alpha_percentile"].quantile(0.25)),
        "avg_selected_hit_rate_real_top20": float(g["selected_hit_rate_real_top20"].mean()),
        "avg_turnover": float(g["buy_turnover"].mean()),
        "avg_trade_cost": float(g["trade_cost"].mean()),
        "avg_board_hhi": float(g["board_hhi"].mean()),
        "avg_chinext_ratio": float(g["board_chinext_ratio"].mean()),
        "avg_star_ratio": float(g["board_star_ratio"].mean()),
        "avg_agreement_overlap": float(g["agreement_overlap"].mean()) if "agreement_overlap" in g.columns else np.nan,
        "avg_agreement_jaccard": float(g["agreement_jaccard"].mean()) if "agreement_jaccard" in g.columns else np.nan,
        "avg_score_rank_corr_exp_roll60": float(g["score_rank_corr_exp_roll60"].mean()) if "score_rank_corr_exp_roll60" in g.columns else np.nan,
    })


rule_summary_all_df = rule_monthly_df.groupby("rule_name").apply(summarize_rule).reset_index(drop=True)
rule_summary_all_df = rule_summary_all_df.sort_values(["cum_ret", "max_drawdown"], ascending=[False, False])

# Fair windows: core where expanding and rolling60 exist; all where rolling72 also exists.
policy_start = score_panel_df.groupby("train_policy")[DATE_COL].min()
core_start = max(policy_start[p] for p in ["expanding_min36", "rolling60m"] if p in policy_start.index)
all_start = max(policy_start[p] for p in ["expanding_min36", "rolling60m", "rolling72m"] if p in policy_start.index)
core_df = rule_monthly_df[rule_monthly_df[DATE_COL] >= core_start].copy()
all_df = rule_monthly_df[rule_monthly_df[DATE_COL] >= all_start].copy()
rule_summary_core_df = core_df.groupby("rule_name").apply(summarize_rule).reset_index(drop=True).sort_values(["cum_ret", "max_drawdown"], ascending=[False, False])
rule_summary_allcommon_df = all_df.groupby("rule_name").apply(summarize_rule).reset_index(drop=True).sort_values(["cum_ret", "max_drawdown"], ascending=[False, False])

# Bad month and agreement diagnostics.
failure_rule_months_df = rule_monthly_df[
    (rule_monthly_df["proxy_net_excess_ret"] < 0)
    & ((rule_monthly_df["random_alpha_percentile"] < 0.20) | (rule_monthly_df.get("agreement_overlap", pd.Series(index=rule_monthly_df.index, dtype=float)) < 3))
].copy().sort_values(["proxy_net_excess_ret", "random_alpha_percentile"])

agreement_diag_df = rule_monthly_df[[DATE_COL, "rule_name", "proxy_net_excess_ret", "random_alpha_percentile", "agreement_overlap", "agreement_jaccard", "score_rank_corr_exp_roll60", "target_count"]].copy()

latest_dt = rule_monthly_df[DATE_COL].max()
latest_targets_df = rule_monthly_df[rule_monthly_df[DATE_COL] == latest_dt].copy()

display(rule_summary_all_df)
display(rule_summary_core_df)
display(rule_summary_allcommon_df)
display(failure_rule_months_df.head(40))
display(latest_targets_df[["rule_name", DATE_COL, "target_count", "proxy_net_excess_ret", "random_alpha_percentile", "agreement_overlap", "board_hhi", "targets"]])


## 可选：JQ-like 日频账户模拟

这段沿用 V61/V65 的账户逻辑：目标列表确定后，调仓日卖出非目标、保留重叠持仓、按现金买入新增目标，逐日用 close 标记净值。

低持仓规则会自然保留现金，所以可以检验“分歧时降仓/空仓”是否真的降低回撤。


In [ ]:
def parse_target_list(x):
    if pd.isnull(x):
        return []
    return [s.strip() for s in str(x).split(",") if s.strip()]


def unique_keep_order(cols):
    seen = set()
    out = []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


def chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def is_kcb_stock(stock):
    return str(stock).startswith(("688", "689"))


def lot_size_for_stock(stock):
    return KCB_MIN_LOT if is_kcb_stock(stock) else NORMAL_MIN_LOT


def floor_to_lot(value, price, lot):
    if pd.isnull(price) or price <= 0 or value <= 0:
        return 0
    return int(np.floor(value / price / lot) * lot)


def calc_buy_cash_cost(amount, price):
    gross = float(amount) * float(price)
    commission = max(MIN_COMMISSION, gross * OPEN_COMMISSION)
    slippage = gross * SLIPPAGE_RATE
    return gross + commission + slippage


def calc_sell_cash_in(amount, price):
    gross = float(amount) * float(price)
    commission = max(MIN_COMMISSION, gross * CLOSE_COMMISSION)
    tax = gross * CLOSE_TAX
    slippage = gross * SLIPPAGE_RATE
    return gross - commission - tax - slippage


def jq_fetch_daily_price_panel(securities, start_date, end_date, fields, chunk_size=120):
    securities = unique_keep_order([s for s in securities if isinstance(s, str) and s])
    if len(securities) == 0:
        return pd.DataFrame()
    parts = []
    chunks_list = list(chunks(securities, chunk_size))
    for sec_chunk in progress_iter(chunks_list, total=len(chunks_list), desc="jq price chunks", leave=False):
        try:
            try:
                df = get_price(sec_chunk, start_date=pd.Timestamp(start_date).strftime("%Y-%m-%d"), end_date=pd.Timestamp(end_date).strftime("%Y-%m-%d"), frequency="daily", fields=fields, skip_paused=False, fq=JQ_SIM_PRICE_FQ, panel=False, fill_paused=True)
            except TypeError:
                df = get_price(sec_chunk, start_date=pd.Timestamp(start_date).strftime("%Y-%m-%d"), end_date=pd.Timestamp(end_date).strftime("%Y-%m-%d"), frequency="daily", fields=fields, skip_paused=False, fq=JQ_SIM_PRICE_FQ, panel=False)
        except NameError:
            raise RuntimeError("JoinQuant get_price is not available in this environment")
        if df is not None and not df.empty:
            parts.append(df.copy())
    if not parts:
        return pd.DataFrame()
    out = pd.concat(parts, axis=0, ignore_index=True, sort=False)
    if "time" in out.columns:
        out["time"] = pd.to_datetime(out["time"]).dt.normalize()
    return out


def jq_fetch_benchmark_close(start_date, end_date):
    try:
        df = get_price(BENCHMARK, start_date=pd.Timestamp(start_date).strftime("%Y-%m-%d"), end_date=pd.Timestamp(end_date).strftime("%Y-%m-%d"), frequency="daily", fields=[JQ_SIM_MARK_FIELD], skip_paused=False, fq=JQ_SIM_PRICE_FQ)
    except NameError:
        raise RuntimeError("JoinQuant get_price is not available in this environment")
    if df is None or len(df) == 0:
        return pd.Series(dtype=float)
    if isinstance(df, pd.Series):
        s = df.copy()
    elif JQ_SIM_MARK_FIELD in df.columns:
        s = df[JQ_SIM_MARK_FIELD].copy()
    else:
        s = df.iloc[:, 0].copy()
    s.index = pd.to_datetime(s.index).normalize()
    return pd.to_numeric(s, errors="coerce").dropna()


def price_matrix(price_df, field):
    if price_df is None or price_df.empty or "time" not in price_df.columns or "code" not in price_df.columns or field not in price_df.columns:
        return pd.DataFrame()
    return price_df.pivot_table(index="time", columns="code", values=field).sort_index()


def get_price_from_row(mat, dt, stock, fallback=np.nan):
    try:
        v = mat.at[pd.Timestamp(dt).normalize(), stock]
    except Exception:
        v = fallback
    if pd.isnull(v):
        return fallback
    return float(v)


def simulate_one_rule_jq_like(rule_df):
    gdf = rule_df.sort_values(DATE_COL).copy()
    if gdf.empty:
        return pd.DataFrame(), pd.DataFrame(), {"jq_like_status": "empty"}
    target_by_date = {}
    all_stocks = []
    for _, row in gdf.iterrows():
        dt = pd.Timestamp(row[DATE_COL]).normalize()
        stocks = parse_target_list(row.get("targets", ""))
        target_by_date[dt] = stocks
        all_stocks.extend(stocks)
    all_stocks = unique_keep_order(all_stocks)
    start_date = pd.Timestamp(gdf[DATE_COL].min()).normalize()
    end_date = pd.Timestamp(gdf["next_date"].dropna().max()).normalize() if "next_date" in gdf.columns and gdf["next_date"].notnull().any() else pd.Timestamp(gdf[DATE_COL].max()).normalize()

    price_df = jq_fetch_daily_price_panel(all_stocks, start_date, end_date, [JQ_SIM_BUY_TIME_FIELD, JQ_SIM_MARK_FIELD], chunk_size=JQ_SIM_CHUNK_SIZE) if all_stocks else pd.DataFrame()
    open_mat = price_matrix(price_df, JQ_SIM_BUY_TIME_FIELD)
    close_mat = price_matrix(price_df, JQ_SIM_MARK_FIELD)
    if close_mat.empty and all_stocks:
        return pd.DataFrame(), pd.DataFrame(), {"jq_like_status": "no_price"}
    if open_mat.empty:
        open_mat = close_mat.copy()
    else:
        open_mat = open_mat.combine_first(close_mat)
    bench_close = jq_fetch_benchmark_close(start_date, end_date)
    trade_dates = list(bench_close.index if close_mat.empty else close_mat.index)
    if len(trade_dates) == 0:
        return pd.DataFrame(), pd.DataFrame(), {"jq_like_status": "no_trade_dates"}

    cash = float(JQ_SIM_INITIAL_CASH)
    positions = {}
    last_mark = {}
    daily_rows = []
    trade_rows = []
    target_dates = set(target_by_date.keys())

    for cur_date in progress_iter(trade_dates, total=len(trade_dates), desc="jq days", leave=False):
        if cur_date in target_dates:
            targets = target_by_date.get(cur_date, [])
            target_set = set(targets)
            for stock in list(positions.keys()):
                amount = positions.get(stock, 0)
                if amount <= 0 or stock in target_set:
                    continue
                px = get_price_from_row(open_mat, cur_date, stock, last_mark.get(stock, np.nan))
                if pd.isnull(px) or px <= 0:
                    continue
                cash_in = calc_sell_cash_in(amount, px)
                cash += cash_in
                trade_rows.append({"date": cur_date, "stock": stock, "side": "sell", "amount": amount, "price": px, "cash_delta": cash_in})
                positions.pop(stock, None)

            current_holds = [s for s, amount in positions.items() if amount > 0]
            target_num = len(targets)
            buy_list = [s for s in targets if positions.get(s, 0) <= 0]
            buy_slots = max(0, target_num - len(current_holds))
            if buy_slots > 0 and cash > 0:
                value_per_slot = cash / float(buy_slots)
                for stock in buy_list:
                    px = get_price_from_row(open_mat, cur_date, stock, get_price_from_row(close_mat, cur_date, stock))
                    if pd.isnull(px) or px <= 0:
                        continue
                    lot = lot_size_for_stock(stock)
                    amount = floor_to_lot(value_per_slot, px * (1.0 + BUY_COST_RATE), lot)
                    if amount < lot:
                        continue
                    cash_cost = calc_buy_cash_cost(amount, px)
                    while amount >= lot and cash_cost > cash:
                        amount -= lot
                        cash_cost = calc_buy_cash_cost(amount, px) if amount > 0 else 0.0
                    if amount < lot or cash_cost <= 0:
                        continue
                    cash -= cash_cost
                    positions[stock] = positions.get(stock, 0) + amount
                    trade_rows.append({"date": cur_date, "stock": stock, "side": "buy", "amount": amount, "price": px, "cash_delta": -cash_cost})

        pos_value = 0.0
        valid_marks = 0
        for stock, amount in list(positions.items()):
            px = get_price_from_row(close_mat, cur_date, stock, last_mark.get(stock, np.nan)) if not close_mat.empty else np.nan
            if pd.isnull(px) or px <= 0:
                continue
            last_mark[stock] = px
            pos_value += float(amount) * float(px)
            valid_marks += 1
        total_value = cash + pos_value
        daily_rows.append({"date": cur_date, "cash": cash, "position_value": pos_value, "total_value": total_value, "position_count": int(len([s for s, a in positions.items() if a > 0])), "valid_marks": int(valid_marks), "is_rebalance_date": bool(cur_date in target_dates)})

    daily_df = pd.DataFrame(daily_rows).sort_values("date")
    daily_df["nav"] = daily_df["total_value"] / float(JQ_SIM_INITIAL_CASH)
    daily_df["drawdown"] = daily_df["nav"] / daily_df["nav"].cummax() - 1.0
    if len(bench_close) > 1:
        bench = bench_close.reindex(daily_df["date"]).ffill().dropna()
        if len(bench) > 1 and bench.iloc[0] > 0:
            bench_nav = bench / bench.iloc[0]
            daily_df = daily_df.merge(pd.DataFrame({"date": bench_nav.index, "benchmark_nav": bench_nav.values}), on="date", how="left")
            daily_df["benchmark_nav"] = daily_df["benchmark_nav"].ffill()
            daily_df["relative_excess_nav"] = daily_df["nav"] / daily_df["benchmark_nav"]
    if "benchmark_nav" not in daily_df.columns:
        daily_df["benchmark_nav"] = np.nan
        daily_df["relative_excess_nav"] = np.nan

    month_rows = []
    prev_value = float(JQ_SIM_INITIAL_CASH)
    prev_bench_nav = 1.0
    for _, row in gdf.iterrows():
        end_dt = pd.Timestamp(row["next_date"]).normalize() if "next_date" in row and not pd.isnull(row.get("next_date")) else pd.Timestamp(row[DATE_COL]).normalize()
        eligible = daily_df[daily_df["date"] <= end_dt]
        if eligible.empty:
            continue
        end_row = eligible.iloc[-1]
        period_ret = float(end_row["total_value"] / prev_value - 1.0) if prev_value > 0 else np.nan
        bench_nav = float(end_row.get("benchmark_nav", np.nan))
        bench_ret = bench_nav / prev_bench_nav - 1.0 if not pd.isnull(bench_nav) and prev_bench_nav > 0 else np.nan
        excess_ret = (1.0 + period_ret) / (1.0 + bench_ret) - 1.0 if not pd.isnull(period_ret) and not pd.isnull(bench_ret) else np.nan
        out = dict(row)
        out.update({"jq_like_period_end": end_row["date"], "jq_like_total_value": float(end_row["total_value"]), "jq_like_nav": float(end_row["nav"]), "jq_like_period_ret": period_ret, "jq_like_benchmark_period_ret": bench_ret, "jq_like_excess_period_ret": excess_ret, "jq_like_drawdown": float(end_row["drawdown"]), "jq_like_position_count": int(end_row["position_count"]), "jq_like_cash": float(end_row["cash"])})
        month_rows.append(out)
        prev_value = float(end_row["total_value"])
        if not pd.isnull(bench_nav):
            prev_bench_nav = bench_nav

    status = {"jq_like_status": "ok", "jq_like_days": int(len(daily_df)), "jq_like_trades": int(len(trade_rows)), "jq_like_final_value": float(daily_df["total_value"].iloc[-1]), "jq_like_cum_ret": float(daily_df["nav"].iloc[-1] - 1.0), "jq_like_max_drawdown": float(daily_df["drawdown"].min())}
    if daily_df["benchmark_nav"].notnull().any():
        status["jq_like_benchmark_cum_ret"] = float(daily_df["benchmark_nav"].dropna().iloc[-1] - 1.0)
        status["jq_like_relative_excess_cum_ret"] = float(daily_df["relative_excess_nav"].dropna().iloc[-1] - 1.0)
    else:
        status["jq_like_benchmark_cum_ret"] = np.nan
        status["jq_like_relative_excess_cum_ret"] = np.nan
    return daily_df, pd.DataFrame(month_rows), status


def maybe_run_jq_like_simulator(rule_monthly):
    if RUN_JQ_LIKE_SIMULATOR == "OFF" or rule_monthly.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    daily_parts = []
    monthly_parts = []
    status_rows = []
    groups = list(rule_monthly.groupby("rule_name"))
    for rule_name, gdf in progress_iter(groups, total=len(groups), desc="jq-like rules"):
        try:
            daily_one, monthly_one, status = simulate_one_rule_jq_like(gdf)
            status.update({"rule_name": rule_name})
            if not daily_one.empty:
                daily_one = daily_one.copy()
                daily_one["rule_name"] = rule_name
                daily_parts.append(daily_one)
            if not monthly_one.empty:
                monthly_one = monthly_one.copy()
                monthly_one["jq_like_available"] = True
                monthly_parts.append(monthly_one)
            status_rows.append(status)
        except RuntimeError as err:
            if RUN_JQ_LIKE_SIMULATOR == "AUTO":
                print("JQ-like skipped:", err)
                return pd.DataFrame(), pd.DataFrame(), pd.DataFrame([{"jq_like_status": str(err)}])
            raise
        except Exception as err:
            status_rows.append({"rule_name": rule_name, "jq_like_status": str(err)})
    daily = pd.concat(daily_parts, ignore_index=True, sort=False) if daily_parts else pd.DataFrame()
    monthly = pd.concat(monthly_parts, ignore_index=True, sort=False) if monthly_parts else pd.DataFrame()
    status = pd.DataFrame(status_rows)
    print("JQ-like daily/monthly/status:", daily.shape, monthly.shape, status.shape)
    return daily, monthly, status


In [ ]:
jq_like_daily_equity_df, jq_like_monthly_df, jq_like_status_df = maybe_run_jq_like_simulator(rule_monthly_df)

jq_like_summary_all_df = pd.DataFrame()
jq_like_summary_core_df = pd.DataFrame()
jq_like_summary_allcommon_df = pd.DataFrame()
if not jq_like_monthly_df.empty and "jq_like_excess_period_ret" in jq_like_monthly_df.columns:
    jq_like_summary_all_df = jq_like_monthly_df.groupby("rule_name").apply(lambda g: summarize_rule(g, "jq_like_excess_period_ret")).reset_index(drop=True).sort_values(["cum_ret", "max_drawdown"], ascending=[False, False])
    jq_core = jq_like_monthly_df[jq_like_monthly_df[DATE_COL] >= core_start].copy()
    jq_all = jq_like_monthly_df[jq_like_monthly_df[DATE_COL] >= all_start].copy()
    jq_like_summary_core_df = jq_core.groupby("rule_name").apply(lambda g: summarize_rule(g, "jq_like_excess_period_ret")).reset_index(drop=True).sort_values(["cum_ret", "max_drawdown"], ascending=[False, False])
    jq_like_summary_allcommon_df = jq_all.groupby("rule_name").apply(lambda g: summarize_rule(g, "jq_like_excess_period_ret")).reset_index(drop=True).sort_values(["cum_ret", "max_drawdown"], ascending=[False, False])
    display(jq_like_summary_all_df)
    display(jq_like_summary_core_df)
    display(jq_like_summary_allcommon_df)
if not jq_like_status_df.empty:
    display(jq_like_status_df)


In [ ]:
rule_monthly_df.to_csv(OUT_DIR / "v66_rule_monthly_proxy.csv", index=False)
rule_summary_all_df.to_csv(OUT_DIR / "v66_rule_summary_all_proxy.csv", index=False)
rule_summary_core_df.to_csv(OUT_DIR / "v66_rule_summary_core_proxy.csv", index=False)
rule_summary_allcommon_df.to_csv(OUT_DIR / "v66_rule_summary_allcommon_proxy.csv", index=False)
failure_rule_months_df.to_csv(OUT_DIR / "v66_failure_rule_months.csv", index=False)
agreement_diag_df.to_csv(OUT_DIR / "v66_agreement_diagnostics.csv", index=False)
latest_targets_df.to_csv(OUT_DIR / "v66_latest_targets_by_rule.csv", index=False)
jq_like_daily_equity_df.to_csv(OUT_DIR / "v66_jq_like_daily_equity.csv", index=False)
jq_like_monthly_df.to_csv(OUT_DIR / "v66_jq_like_monthly.csv", index=False)
jq_like_status_df.to_csv(OUT_DIR / "v66_jq_like_status.csv", index=False)
jq_like_summary_all_df.to_csv(OUT_DIR / "v66_rule_summary_all_jq_like.csv", index=False)
jq_like_summary_core_df.to_csv(OUT_DIR / "v66_rule_summary_core_jq_like.csv", index=False)
jq_like_summary_allcommon_df.to_csv(OUT_DIR / "v66_rule_summary_allcommon_jq_like.csv", index=False)

print("saved outputs:")
for p in sorted(OUT_DIR.glob("v66_*.csv")):
    print("-", p)


## 怎么读 V66

优先看：

1. `v66_rule_summary_core_jq_like.csv`：2024-2026 的核心共同区间，比较 expanding / rolling60 / ensemble / 防守规则。
2. `v66_rule_summary_allcommon_jq_like.csv`：2025-2026，包含 rolling72 和 all-available blend。
3. `v66_rule_summary_all_jq_like.csv`：各规则能跑的全区间，但注意起始时间不同。
4. `v66_failure_rule_months.csv`：看 ensemble / 防守规则有没有减少 2025-03、2025-05、2026-03 的亏损。
5. `v66_agreement_diagnostics.csv`：看 overlap / score correlation 是否能作为健康监测指标。

判断标准：

- 如果 ensemble 的 JQ-like 超额高于单方法，同时回撤更小，可以进入主线。
- 如果防守规则降低回撤但收益损失过大，不作为默认，只作为风险开关。
- 如果分歧度低的月份显著更差，agreement overlap 可以进入健康监测。
- 如果所有 ensemble 都弱于 expanding/rolling60，说明暂时不要复杂化，保持单规则。
